# Ping Envelope fall

One continuous one-pole chase toward silence. No attack, no hold, no multipoint ADSR.

```text
env[0] = trigger_height
x = feedback * env + offset
shape = clip(10 ** (5 * (x - 1)), 0, 1) * max(sign(x), 0)
f = shape * max_release_hz
u = max(sign(sample_rate / 2 - f), 0)
k = u * clip(1 - exp(-2 * pi * f / sample_rate), 0, 1) + (1 - u)
env = env + k * (0 - env)
```

`clip(v, 0, 1) = min(max(v, 0), 1)` and `sign(0) = 0`. The 5-decade shape is the fixed Ping Envelope release map, shared by every curve. There is no mode axis. Example overlays are saved sets of the four sliders.

| Example | Feedback | Max release Hz | Offset | Trigger height |
| --- | --- | --- | --- | --- |
| A | 1 | 10 | 0 | 1 |
| B | 0.7718 | 1000 | 0.5 | 1 |
| C | 0.7718 | 1000 | 0 | 1 |

The loop stops at amplitude **0.001** (−60 dB). **0.01** (−40 dB) is only a dashed marker. Sample rate is 48 kHz. The interactive sliders and example checkboxes are in `index.html` (run `python -m graphs.ping_envelope` from the repo root).

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in [here, *here.parents]:
    if (cand / "graphs" / "ping_envelope" / "simulate.py").exists():
        sys.path.insert(0, str(cand))
        break
else:
    raise SystemExit("Open this notebook from the soemdsp-research repo.")

from graphs.ping_envelope.simulate import examples, load_config, simulate_fall

config = load_config()
print(f"sample rate {config['sample_rate']} Hz, eng floor {config['eng_floor']}")
for example in examples():
    curve = simulate_fall(example.params)
    print(
        f"{example.label:18} stop={curve.stopped:10} "
        f"-40dB={curve.t_perceptual_s}  -60dB={curve.t_eng_s}  "
        f"end={curve.end_amplitude:.6f}"
    )

In [ ]:
from graphs.ping_envelope.plot import build_figure

named = []
for example in examples():
    named.append((example.label, example.color, simulate_fall(example.params)))

fig = build_figure(named, log_y=True, title="Ping Envelope fall — example settings")
fig.show()